# Chopp & Cia · 08 — Comparação de Cenários

**Projeto Integrador VI** · FATEC Votorantim · 2º Semestre/2026

Os notebooks 05, 06 e 07 respondem *"qual hiperparâmetro é melhor para este
modelo, nesta população"*. Este responde a pergunta de fora: **quais decisões —
de população, de alvo, de features e de modelo — produzem o melhor resultado, e
quanto cada uma delas vale?**

| | |
|:---|:---|
| **Entrada** | `dataset_consolidado_v<DATA_VERSION>.csv` (notebook 01) ou a tabela do 02 |
| **Saída** | `cenarios_v<DATA_VERSION>.csv` — uma linha por combinação avaliada |
| **Ambiente** | Windows local ou Databricks |

### Por que este notebook existe

Um resultado de modelo não é um número, é um número **condicionado** a escolhas
anteriores: quem entrou na base, o que se chamou de risco, quais colunas o modelo
viu. Trocar qualquer uma delas move a métrica — às vezes mais do que trocar o
algoritmo.

Enquanto essas escolhas ficam espalhadas pelos painéis de cinco notebooks, comparar
duas rodadas exige confiar na memória de quem rodou. Aqui elas viram **eixos de uma
grade**, e a comparação vira tabela.

> **O que este notebook NÃO faz:** ele não elege o modelo de produção. A varredura
> usa configuração fixa por modelo, para que a comparação seja entre *cenários*,
> não entre ajustes finos. Escolhido o cenário, a otimização de hiperparâmetro
> continua sendo trabalho dos notebooks 05-07.

## 1. Painel de controle

Cada bloco abaixo é um **eixo** da grade. O total de execuções é o produto dos
eixos — o fim do painel calcula e imprime esse número antes de rodar qualquer
coisa.

Acrescentar um cenário é acrescentar uma entrada na lista; nada mais muda.

In [ ]:
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# ── Entrada e saída ────────────────────────────────────────────────
CAMINHO_CSV = r""
PASTA_SAIDA = r""
TABELA = "projetointegrador.projetointegrador.dataset_consolidado_v1_0"   # Databricks

DATA_VERSION = "1.0"
SOBRESCREVER = False
RANDOM_STATE = 42          # semente única, propagada à CV e a todos os estimadores

try:
    spark                                                      # noqa: F821
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False

# ── MLflow: um experimento por POPULAÇÃO × ALVO ──────────────────────────
# Granularidade deliberada. Um experimento é o lugar onde comparar runs faz
# sentido, e comparar MCC só é legítimo dentro do MESMO problema — mesma
# população, mesmo alvo. Variar features e modelo DENTRO dele é a comparação
# que interessa; variar o alvo entre experimentos mantém separadas as
# dificuldades diferentes.
#
#   Chopp_08__core-todos__atraso-20-OU     ← experimento
#     ├── completo__Regressao_Logistica    ← run
#     ├── completo__Arvore_de_Decisao
#     ├── sem-MEDIA__Regressao_Logistica
#     └── completo__baseline
MLFLOW_ATIVO = True         # só tem efeito no Databricks; local é ignorado
MLFLOW_RAIZ = "/Users/brunofsaraujo@hotmail.com/Chopp_Cia_Cenarios"
MLFLOW_LOG_MODELO = False   # False: só métricas/params/tags. Ligar sobe 216
                             # artefatos de modelo ao Volume — lento e raramente
                             # útil aqui, já que o ajuste fino é dos nb 05-07.

# ── Pacote de produção — o único do pipeline ──────────────────────────────
# Os notebooks 05-07 são estudos: variam hiperparâmetro com população, alvo e
# features FIXOS, e por isso não promovem nada a produção. O campeão de um
# estudo de hiperparâmetro é o melhor ajuste dentro de um cenário que talvez
# nem seja o melhor cenário. Só aqui, depois de varrer os quatro eixos, existe
# base para dizer "este é o modelo".
EMPACOTAR_CAMPEAO = True

CAMPEAO_PRODUCAO = "sem_vazamento"
# "sem_vazamento"    : melhor MCC entre os conjuntos que excluem MEDIA_DIAS_ATRASO.
#                       É o número honesto — o padrão, e o que a seção "RESPOSTA
#                       DA GRADE" recomenda citar como capacidade preditiva.
# "absoluto"         : maior MCC da grade inteira. Mede aderência à regra de
#                       negócio, não previsão de conduta — não promova sem saber.
# "cobertura_maxima" : maior MCC entre os que retêm mais clientes, sem vazamento.

TEST_SIZE_PRODUCAO = 0.30   # mesmo holdout do notebook 04, para que a métrica de
                             # teste do pacote seja comparável à dos estudos.
NOME_REGISTRADO = "workspace.default.chopp_risco_campeao"   # Databricks

if EM_DATABRICKS and MLFLOW_ATIVO:
    import re as _re
    import mlflow
    import mlflow.sklearn   # no topo: importar dentro da função tornaria o nome
                             # "mlflow" local a ela, sombreando este global.

    def slug(texto):
        """Nome de experimento aceito pelo Databricks: sem espaço, sem ·, sem %."""
        t = (str(texto).replace("·", "-").replace("%", "pct")
             .replace("ç", "c").replace("ã", "a").replace("á", "a")
             .replace("é", "e").replace("ê", "e").replace("í", "i")
             .replace("ó", "o").replace("ô", "o").replace("ú", "u"))
        t = _re.sub(r"[^A-Za-z0-9]+", "-", t).strip("-").lower()
        return _re.sub(r"-+", "-", t)

# ── EIXO 1 · POPULAÇÃO — quem entra na base ────────────────────────────
# core        : recorte de negócio (comprou chopp/chopeira)
# min_compras : exclusivo — 0 mantém quem tem 1+ compra
# janela_dias : None = sem recorte temporal; 365 = comprou no último ano
CENARIOS_POPULACAO = [
    {"nome": "core · todos",       "core": True,  "min_compras": 0, "janela_dias": None},
    {"nome": "core · 2+ compras",  "core": True,  "min_compras": 1, "janela_dias": None},
    {"nome": "core · 3+ compras",  "core": True,  "min_compras": 2, "janela_dias": None},
    {"nome": "carteira inteira",   "core": False, "min_compras": 0, "janela_dias": None},
    {"nome": "core · ativos 365d", "core": True,  "min_compras": 0, "janela_dias": 365},
    {"nome": "core · ativos 180d", "core": True,  "min_compras": 0, "janela_dias": 180},
]

# ── EIXO 2 · ALVO — o que se chama de risco ─────────────────────────────
# Regra de negócio, não estatística: cada linha aqui é um problema DIFERENTE.
# Comparar MCC entre alvos distintos compara dificuldades distintas — por isso a
# tabela final agrupa por alvo antes de ranquear.
CENARIOS_ALVO = [
    {"nome": "atraso > 20% (OU)", "limite": 0.20, "combinador": "OU"},
    {"nome": "atraso > 20% (E)",  "limite": 0.20, "combinador": "E"},
    {"nome": "atraso > 30% (OU)", "limite": 0.30, "combinador": "OU"},
    {"nome": "atraso > 50% (OU)", "limite": 0.50, "combinador": "OU"},
]

# ── EIXO 3 · FEATURES — o que o modelo vê ─────────────────────────────
FEATURES_NUM = [
    "FREQUENCIA_COMPRAS", "TICKET_MEDIO", "TOTAL_GASTO",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "TOTAL_PARCELAS", "TOTAL_COMODATOS",
    "PCT_COMPRAS_A_PRAZO", "PRAZO_MEDIO_COMODATO", "VALOR_MEDIO_COMODATO",
    "MEDIA_DIAS_ATRASO_PAG", "MEDIA_DIAS_ATRASO_COM",
]
FEATURES_CAT = ["PERFIL", "CIDADE", "PAGAMENTO"]

# "sem MEDIA_DIAS_ATRASO" é o cenário mais importante da grade: essas duas colunas
# compartilham origem aritmética com o alvo. A diferença entre os dois conjuntos
# é a MEDIDA do vazamento — não uma curiosidade.
CENARIOS_FEATURES = [
    {"nome": "completo",              "remover": []},
    {"nome": "sem MEDIA_DIAS_ATRASO", "remover": ["MEDIA_DIAS_ATRASO_PAG",
                                                  "MEDIA_DIAS_ATRASO_COM"]},
    {"nome": "só cadastro e RFM",     "remover": ["MEDIA_DIAS_ATRASO_PAG",
                                                  "MEDIA_DIAS_ATRASO_COM",
                                                  "TOTAL_PARCELAS", "TOTAL_COMODATOS",
                                                  "PRAZO_MEDIO_COMODATO",
                                                  "VALOR_MEDIO_COMODATO"]},
]

# ── EIXO 4 · MODELO ─────────────────────────────────────────────
# Configuração FIXA por modelo, deliberadamente. A grade compara cenários; se cada
# célula também otimizasse hiperparâmetro, não se saberia se a diferença veio do
# cenário ou do ajuste. O ajuste fino continua nos notebooks 05-07.
CENARIOS_MODELO = [
    {"nome": "Regressão Logística", 
     "classe": LogisticRegression, 
     "hp": {"C": 1.0, 
            "penalty": "l2", 
            "solver": "lbfgs", 
            "max_iter": 1000,
            "class_weight": "balanced"}
    },
    {"nome": "Árvore de Decisão", 
     "classe": DecisionTreeClassifier,
     "hp": {"max_depth": 5, 
            "min_samples_leaf": 5, 
            "class_weight": "balanced"}
    },
    {"nome": "Random Forest", 
     "classe": RandomForestClassifier,
     "hp": {"n_estimators": 200, 
            "max_depth": 8, 
            "min_samples_leaf": 2,
            "class_weight": "balanced", 
            "n_jobs": -1}
    },
]
BASELINE_ATIVO = True    # DummyClassifier(most_frequent): MCC = 0 por construção

# ── Validação e viabilidade ──────────────────────────────────────
CV = {"n_splits": 5, "shuffle": True}
SCORING = {
    "mcc": "matthews_corrcoef", "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc", "average_precision": "average_precision",
    "recall": "recall", "precision": "precision", "f1": "f1",
}
METRICA_RANKING = "mcc"    # só sobe quando AMBAS as classes são bem classificadas

# Um cenário com classe rara pequena demais não é comparável: a métrica vira ruido
# de sorteio. Em vez de omiti-lo, a grade o marca como INVIÁVEL — descobrir que um
# recorte não sustenta modelo É um resultado.
MIN_CASOS_RAROS_POR_FOLD = 5

CSV_SAIDA = {"sep": ";", "encoding": "utf-8-sig", "index": False}

plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.grid": True, "grid.color": "#e3e2df",
                     "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False})

N_TOTAL = (len(CENARIOS_POPULACAO) * len(CENARIOS_ALVO)
           * len(CENARIOS_FEATURES) * len(CENARIOS_MODELO))
print(f"Ambiente  : {'Databricks' if EM_DATABRICKS else 'Local'}")
print(f"Eixos     : {len(CENARIOS_POPULACAO)} populações × {len(CENARIOS_ALVO)} alvos × "
      f"{len(CENARIOS_FEATURES)} conjuntos × {len(CENARIOS_MODELO)} modelos")
print(f"Execuções : {N_TOTAL:,} ajustes de CV ({CV['n_splits']} folds cada)")

## 2. Carga

Este notebook lê o **consolidado**, não o split do 04 — de propósito. A população
é um dos eixos que ele varre, e o split já tem uma população escolhida embutida.

O tratamento de ausentes replica exatamente o do notebook 04: zero para contagem,
mediana para `TICKET_MEDIO`. Se divergisse, os resultados daqui não seriam
comparáveis com os de lá.

In [ ]:
if EM_DATABRICKS:
    dados = spark.table(TABELA).toPandas()                     # noqa: F821
    origem = TABELA
else:
    if not CAMINHO_CSV.strip():
        raise ValueError("Preencha CAMINHO_CSV com o consolidado gerado pelo notebook 01.")
    caminho = Path(CAMINHO_CSV.strip())
    if not caminho.exists():
        raise FileNotFoundError(f"CSV não encontrado: {caminho}")
    dados = pd.read_csv(caminho, sep=";", encoding="utf-8-sig", low_memory=False)
    origem = caminho.name

dados.columns = [str(c).strip().upper() for c in dados.columns]

OUTPUT_DIR = (
    Path(PASTA_SAIDA.strip()) if PASTA_SAIDA.strip()
    else (Path(CAMINHO_CSV).parent if not EM_DATABRICKS else Path("."))
)
if not EM_DATABRICKS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DESTINO = OUTPUT_DIR / f"cenarios_v{DATA_VERSION.replace('.', '_')}.csv"

# Colunas do alvo: não são feature em nenhum cenário, mas são necessárias para
# CONSTRUIR o alvo de cada linha do eixo 2.
COLS_ALVO = ["TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO"]
COLS_POP = ["CORE_BUSINESS", "FREQUENCIA_COMPRAS", "DIAS_DESDE_ULTIMA_COMPRA"]

faltando = sorted(set(FEATURES_NUM + FEATURES_CAT + COLS_ALVO + COLS_POP)
                  - set(dados.columns))
if faltando:
    raise KeyError(f"Colunas ausentes em {origem}: {faltando}")

for coluna in set(FEATURES_NUM + COLS_ALVO + COLS_POP):
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
for coluna in FEATURES_CAT:
    dados[coluna] = dados[coluna].fillna("NÃO INFORMADO")

# Mesmo tratamento do notebook 04 — se divergisse, os números não conversariam.
COLS_ZERO = [c for c in FEATURES_NUM if c != "TICKET_MEDIO"]
dados[COLS_ZERO] = dados[COLS_ZERO].fillna(0)
dados["TICKET_MEDIO"] = dados["TICKET_MEDIO"].fillna(dados["TICKET_MEDIO"].median())

versao_lida = (str(dados["_DATA_VERSION"].iloc[0])
               if "_DATA_VERSION" in dados.columns else "desconhecida")
if versao_lida not in ("desconhecida", DATA_VERSION):
    print(f"ATENÇÃO: o painel diz DATA_VERSION={DATA_VERSION}, o arquivo diz "
          f"{versao_lida}. Os cenários sairão rotulados com a do painel.")

print(f"Origem   : {origem}")
print(f"Carteira : {len(dados):,} clientes × {dados.shape[1]} colunas")
print(f"Destino  : {CSV_DESTINO.name}")

## 3. Motor de cenários

Quatro funções pequenas, uma por eixo, e uma que avalia a combinação. Ficam
separadas porque cada eixo tem uma regra própria — e porque um eixo novo (um
recorte por região, por exemplo) precisa mexer em uma função só.

A ordem importa: a população é recortada **antes** de o alvo ser calculado. Um
cliente que sai pela janela temporal não deve influenciar a prevalência do
cenário — se o alvo viesse primeiro, ele influenciaria.

In [ ]:
def aplicar_populacao(df, cenario):
    """Eixo 1: recorta QUEM entra. Devolve uma cópia."""
    sub = df
    if cenario["core"]:
        sub = sub[sub["CORE_BUSINESS"] == 1]
    sub = sub[sub["FREQUENCIA_COMPRAS"] > cenario["min_compras"]]
    if cenario["janela_dias"] is not None:
        # Quem nunca comprou tem DIAS_DESDE_ULTIMA_COMPRA nulo e cai fora da
        # janela por definição — o NaN não sobrevive à comparação, o que é a
        # leitura correta: sem compra não há atividade recente.
        sub = sub[sub["DIAS_DESDE_ULTIMA_COMPRA"] <= cenario["janela_dias"]]
    return sub.copy()


def construir_alvo(df, cenario):
    """Eixo 2: define O QUE é risco. A regra é de negócio, não estatística."""
    atraso_pag = df["TAXA_ATRASO_PAGAMENTO"] > cenario["limite"]
    atraso_com = df["TAXA_ATRASO_COMODATO"] > cenario["limite"]
    condicao = (atraso_pag | atraso_com) if cenario["combinador"] == "OU" \
        else (atraso_pag & atraso_com)
    return condicao.astype(int)


def montar_features(cenario):
    """Eixo 3: quais COLUNAS o modelo vê."""
    remover = set(cenario["remover"])
    num = [c for c in FEATURES_NUM if c not in remover]
    cat = [c for c in FEATURES_CAT if c not in remover]
    return num, cat


def montar_pipeline(estimador, num, cat):
    """Pré-processamento idêntico ao dos notebooks 05-07: sem isso a comparação
    entre este notebook e aqueles mediria o pré-processamento, não o cenário."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), num),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
        ])),
        ("est", estimador),
    ])


def instanciar(cenario_modelo):
    hp = dict(cenario_modelo["hp"])
    if "random_state" in cenario_modelo["classe"]().get_params():
        hp["random_state"] = RANDOM_STATE
    return cenario_modelo["classe"](**hp)


def avaliar(X, y, num, cat, estimador):
    """CV estratificada. Devolve média e desvio de cada métrica do SCORING."""
    cv = StratifiedKFold(n_splits=CV["n_splits"], shuffle=CV["shuffle"],
                         random_state=RANDOM_STATE)
    resultado = cross_validate(montar_pipeline(estimador, num, cat), X, y,
                               cv=cv, scoring=SCORING, n_jobs=1,
                               error_score="raise")
    saida = {}
    for metrica in SCORING:
        valores = resultado[f"test_{metrica}"]
        saida[metrica] = float(np.mean(valores))
        saida[f"{metrica}_dp"] = float(np.std(valores))
    return saida


# ── Registro no MLflow ───────────────────────────────────────────────────
# Um experimento por população × alvo; um run por conjunto de features × modelo.
# Fora do Databricks (ou com MLFLOW_ATIVO = False) as duas funções não fazem
# nada — o notebook roda igual, e o CSV continua sendo a saída canônica.

_experimentos = {}   # cache: (populacao, alvo) → experiment_id


def _experimento_de(cen_pop, cen_alvo):
    """Cria (ou recupera) o experimento desta combinação população × alvo."""
    chave = (cen_pop["nome"], cen_alvo["nome"])
    if chave not in _experimentos:
        nome = f"{MLFLOW_RAIZ}__{slug(cen_pop['nome'])}__{slug(cen_alvo['nome'])}"
        existente = mlflow.get_experiment_by_name(nome)
        _experimentos[chave] = (existente.experiment_id if existente
                                else mlflow.create_experiment(nome))
    return _experimentos[chave]


def _tags(cen_pop, cen_alvo, base):
    """Tags iguais em todos os runs: permitem filtrar a grade inteira na UI."""
    return {
        "eixo.populacao": cen_pop["nome"],
        "eixo.alvo": cen_alvo["nome"],
        "pop.core": str(cen_pop["core"]),
        "pop.min_compras": str(cen_pop["min_compras"]),
        "pop.janela_dias": str(cen_pop["janela_dias"]),
        "alvo.limite": str(cen_alvo["limite"]),
        "alvo.combinador": cen_alvo["combinador"],
        "data_version": DATA_VERSION,
        "notebook": "08_cenarios",
    }


def registrar_run(cen_pop, cen_alvo, cen_feat, nome_modelo, hp, num, cat, base, metricas):
    if not (EM_DATABRICKS and MLFLOW_ATIVO):
        return
    with mlflow.start_run(experiment_id=_experimento_de(cen_pop, cen_alvo),
                          run_name=f"{slug(cen_feat['nome'])}__{slug(nome_modelo)}"):
        mlflow.set_tags({**_tags(cen_pop, cen_alvo, base),
                         "eixo.features": cen_feat["nome"],
                         "eixo.modelo": nome_modelo,
                         "viavel": "True"})
        mlflow.log_params({
            **{f"hp.{k}": v for k, v in hp.items()},
            "features.conjunto": cen_feat["nome"],
            "features.removidas": ",".join(cen_feat["remover"]) or "(nenhuma)",
            "features.n": len(num) + len(cat),
            "cv.n_splits": CV["n_splits"],
            "random_state": RANDOM_STATE,
        })
        # n_clientes/prevalencia são métricas, não params: assim entram nos
        # gráficos de comparação da UI junto com o MCC.
        mlflow.log_metrics({
            **{k: v for k, v in metricas.items()},
            "n_clientes": float(base["n_clientes"]),
            "prevalencia": float(base["prevalencia"]),
            "classe_rara": float(base["classe_rara"]),
            "raros_por_fold": float(base["raros_por_fold"]),
        })
        if MLFLOW_LOG_MODELO:
            modelo = next((m for m in CENARIOS_MODELO if m["nome"] == nome_modelo), None)
            if modelo is not None:
                pipe = montar_pipeline(instanciar(modelo), num, cat)
                pipe.fit(universo[num + cat], construir_alvo(universo, cen_alvo))
                mlflow.sklearn.log_model(sk_model=pipe, name="modelo")


def registrar_inviavel(cen_pop, cen_alvo, base, motivo):
    """Run sem métricas de modelo: guarda POR QUE o cenário não foi avaliado."""
    if not (EM_DATABRICKS and MLFLOW_ATIVO):
        return
    with mlflow.start_run(experiment_id=_experimento_de(cen_pop, cen_alvo),
                          run_name="INVIAVEL"):
        mlflow.set_tags({**_tags(cen_pop, cen_alvo, base),
                         "viavel": "False", "motivo": motivo})
        mlflow.log_metrics({
            "n_clientes": float(base["n_clientes"]),
            "prevalencia": float(base["prevalencia"]),
            "classe_rara": float(base["classe_rara"]),
            "raros_por_fold": float(base["raros_por_fold"]),
        })


print("Motor pronto: 4 eixos, pré-processamento igual ao dos notebooks 05-07.")
if EM_DATABRICKS and MLFLOW_ATIVO:
    n_exp = len(CENARIOS_POPULACAO) * len(CENARIOS_ALVO)
    print(f"MLflow   : {n_exp} experimentos sob {MLFLOW_RAIZ}__*")
    print(f"           modelo por run: {'sim' if MLFLOW_LOG_MODELO else 'não (só métricas)'}")
else:
    print("MLflow   : inativo (fora do Databricks) — a saída é o CSV.")

## 3A. Como a grade aparece no MLflow

No Databricks, cada combinação **população × alvo** vira um experimento, e cada
**conjunto de features × modelo** vira um run dentro dele.

```
Chopp_Cia_Cenarios__core-todos__atraso-20-ou      ← experimento
  ├── completo__regressao-logistica               ← run   MCC 0.41
  ├── completo__arvore-de-decisao                 ← run   MCC 0.38
  ├── completo__random-forest                     ← run   MCC 0.44
  ├── completo__baseline                          ← run   MCC 0.00
  ├── sem-media-dias-atraso__regressao-logistica  ← run   MCC 0.22
  └── ...
Chopp_Cia_Cenarios__ativos-365d__atraso-20-ou     ← outro experimento
  └── ...
```

**Por que essa divisão.** O experimento é a unidade em que a UI do MLflow compara
runs — ordena por métrica, sobrepõe gráficos, mostra a tabela lado a lado. Isso só
é legítimo entre runs que respondem à *mesma* pergunta. População e alvo definem a
pergunta: mudar o alvo muda a dificuldade do problema, e um MCC de 0,44 num alvo
não é "melhor" que 0,38 em outro — são escalas diferentes.

Features e modelo, ao contrário, são exatamente o que se quer comparar com o
problema fixo. Por isso ficam como runs no mesmo experimento.

Todo run carrega as mesmas **tags** (`eixo.populacao`, `eixo.alvo`,
`eixo.features`, `eixo.modelo`, `viavel`), então a grade inteira continua
recuperável de uma vez quando a pergunta for global:

```python
mlflow.search_runs(experiment_names=[...], filter_string="tags.eixo.features = 'completo'")
```

Cenários inviáveis também viram run — com nome `INVIAVEL` e a tag `motivo`.
Um recorte que não sustenta validação é resultado registrado, não lacuna.

> O CSV continua sendo a saída canônica: fora do Databricks nada disso roda, e o
> notebook produz o mesmo `cenarios_v1_0.csv`.


## 4. Execução da grade

Percorre população → alvo → features → modelo, nessa ordem, porque é a ordem das
dependências: o alvo depende da população, e as features e o modelo dependem dos
dois.

Antes de treinar qualquer coisa, cada par (população, alvo) passa pelo teste de
viabilidade. Um cenário com menos de `MIN_CASOS_RAROS_POR_FOLD` casos raros por
fold é registrado com o motivo e **não** é treinado: uma métrica calculada sobre
2 casos não é um resultado ruim, é um resultado sem significado — e misturar as
duas coisas na mesma tabela é pior do que omitir.

In [ ]:
registros = []
inicio = time.time()

for cen_pop in CENARIOS_POPULACAO:
    universo = aplicar_populacao(dados, cen_pop)

    for cen_alvo in CENARIOS_ALVO:
        y = construir_alvo(universo, cen_alvo)
        n_pos, n_neg = int(y.sum()), int((y == 0).sum())
        minoria = min(n_pos, n_neg)
        por_fold = minoria / CV["n_splits"]

        base = {
            "populacao": cen_pop["nome"], "alvo": cen_alvo["nome"],
            "n_clientes": len(universo), "prevalencia": float(y.mean()) if len(y) else np.nan,
            "classe_rara": minoria, "raros_por_fold": round(por_fold, 1),
        }

        if minoria < MIN_CASOS_RAROS_POR_FOLD * CV["n_splits"]:
            motivo = (f"classe rara com {minoria} caso(s) — "
                      f"{por_fold:.1f} por fold, mínimo {MIN_CASOS_RAROS_POR_FOLD}")
            registros.append({**base, "features": "—", "modelo": "—",
                              "viavel": False, "motivo": motivo})
            # O cenário inviável também é registrado: descobrir que um recorte
            # não sustenta modelo é resultado, não ausência de resultado.
            registrar_inviavel(cen_pop, cen_alvo, base, motivo)
            continue

        for cen_feat in CENARIOS_FEATURES:
            num, cat = montar_features(cen_feat)
            X = universo[num + cat]

            for cen_modelo in CENARIOS_MODELO:
                metricas = avaliar(X, y, num, cat, instanciar(cen_modelo))
                registros.append({
                    **base, "features": cen_feat["nome"], "modelo": cen_modelo["nome"],
                    "n_features": len(num) + len(cat),
                    "viavel": True, "motivo": "", **metricas,
                })
                registrar_run(cen_pop, cen_alvo, cen_feat, cen_modelo["nome"],
                              cen_modelo["hp"], num, cat, base, metricas)

            if BASELINE_ATIVO:
                # Âncora: prever sempre a classe majoritária. MCC = 0 por
                # construção. Um modelo que não supera isso não aprendeu nada.
                metricas = avaliar(X, y, num, cat,
                                   DummyClassifier(strategy="most_frequent"))
                registros.append({
                    **base, "features": cen_feat["nome"], "modelo": "· baseline",
                    "n_features": len(num) + len(cat),
                    "viavel": True, "motivo": "", **metricas,
                })
                registrar_run(cen_pop, cen_alvo, cen_feat, "· baseline",
                              {"strategy": "most_frequent"}, num, cat, base, metricas)

resultados = pd.DataFrame(registros)
duracao = time.time() - inicio

n_viavel = int(resultados["viavel"].sum())
n_inviavel = int((~resultados["viavel"]).sum())
print(f"{len(resultados):,} linhas em {duracao:.0f}s "
      f"({n_viavel:,} avaliadas · {n_inviavel} cenários inviáveis)")

## 5. Cenários inviáveis

A primeira leitura da grade não é sobre qual modelo ganhou — é sobre quais
perguntas esta base **não** consegue responder.

Um recorte que deixa a classe rara com poucos casos não produz métrica confiável
em nenhum algoritmo. Saber disso antes de comparar evita a conclusão mais comum
e mais errada deste tipo de estudo: eleger um campeão que venceu por sorteio.

In [ ]:
inviaveis = resultados[~resultados["viavel"]]

if inviaveis.empty:
    print("Todos os cenários da grade têm classe rara suficiente para validação.")
else:
    print(f"{len(inviaveis)} combinação(ões) de população × alvo não sustentam validação:\n")
    for _, linha in inviaveis.iterrows():
        print(f"  {linha['populacao']:<20} {linha['alvo']:<20} "
              f"n={linha['n_clientes']:>5,}  {linha['motivo']}")

    print("\nEstes cenários seguem no CSV com viavel=False — o registro de que "
          "foram testados\ne descartados por desenho, não por esquecimento.")

## 6. O ranking

Ordenado por MCC de validação cruzada. MCC, e não acurácia ou AUC, porque só ele
sobe quando **ambas** as classes são bem classificadas — numa base com ~90% de
prevalência, um modelo que chuta "todo mundo é risco" acerta 90% e tem MCC = 0.

O baseline aparece na tabela para tornar esse piso visível.

In [ ]:
viaveis = resultados[resultados["viavel"]].copy()
ranking = viaveis.sort_values(METRICA_RANKING, ascending=False)

COLS_VISAO = ["populacao", "alvo", "features", "modelo", "n_clientes",
              "classe_rara", "prevalencia", METRICA_RANKING,
              "balanced_accuracy", "roc_auc", "recall"]

def formatar(df):
    saida = df[COLS_VISAO].copy()
    saida["prevalencia"] = (saida["prevalencia"] * 100).map("{:.1f}%".format)
    saida["n_clientes"] = saida["n_clientes"].map("{:,}".format)
    for c in [METRICA_RANKING, "balanced_accuracy", "roc_auc", "recall"]:
        saida[c] = saida[c].map("{:.3f}".format)
    return saida

print("=" * 100)
print(f"{'TOP 15 CENÁRIOS (por MCC de validação cruzada)':^100}")
print("=" * 100)
print(formatar(ranking.head(15)).to_string(index=False))

melhor = ranking.iloc[0]
print(f"\nMelhor combinação: {melhor['populacao']} · {melhor['alvo']} · "
      f"{melhor['features']} · {melhor['modelo']}")
print(f"  MCC {melhor[METRICA_RANKING]:.3f} (±{melhor[METRICA_RANKING + '_dp']:.3f}) · "
      f"AUC {melhor['roc_auc']:.3f} · recall {melhor['recall']:.3f}")

baselines = viaveis[viaveis["modelo"] == "· baseline"]
if not baselines.empty:
    print(f"\nBaseline (classe majoritária): MCC médio {baselines[METRICA_RANKING].mean():.3f}, "
          f"acurácia balanceada {baselines['balanced_accuracy'].mean():.3f}")
    acima = viaveis[(viaveis["modelo"] != "· baseline")
                    & (viaveis[METRICA_RANKING] > 0.05)]
    print(f"{len(acima)} de {len(viaveis[viaveis['modelo'] != '· baseline'])} "
          "configurações superam o baseline de forma não trivial.")

## 7. Quanto vale cada eixo

O ranking diz qual combinação ganhou. Esta seção diz **por quê** — qual eixo
move a métrica e qual é quase indiferente.

A leitura é comparativa dentro de cada eixo: fixados os outros três, quanto varia
o MCC ao trocar só este? Um eixo com variação grande é uma decisão que merece
discussão de negócio; um eixo achatado é uma escolha que pode ser feita por
conveniência.

In [ ]:
modelos_reais = viaveis[viaveis["modelo"] != "· baseline"]

print("=" * 76)
print(f"{'EFEITO DE CADA EIXO SOBRE O MCC':^76}")
print("=" * 76)

for eixo, rotulo in [("populacao", "POPULAÇÃO"), ("alvo", "ALVO"),
                     ("features", "FEATURES"), ("modelo", "MODELO")]:
    resumo = (modelos_reais.groupby(eixo)[METRICA_RANKING]
              .agg(["mean", "max", "count"]).sort_values("mean", ascending=False))
    amplitude = resumo["mean"].max() - resumo["mean"].min()
    print(f"\n{rotulo}  (amplitude entre médias: {amplitude:.3f})")
    print(f"  {'':<26}{'MCC médio':>11}{'melhor':>9}{'n':>6}")
    for nome, linha in resumo.iterrows():
        print(f"  {str(nome):<26}{linha['mean']:>11.3f}{linha['max']:>9.3f}"
              f"{int(linha['count']):>6}")

amplitudes = {}
for eixo in ["populacao", "alvo", "features", "modelo"]:
    medias = modelos_reais.groupby(eixo)[METRICA_RANKING].mean()
    amplitudes[eixo] = float(medias.max() - medias.min())

ordem = sorted(amplitudes.items(), key=lambda kv: kv[1], reverse=True)
print("\n" + "=" * 76)
print("Eixos ordenados por quanto movem a métrica:")
for i, (eixo, amp) in enumerate(ordem, 1):
    print(f"  {i}. {eixo:<14} {amp:.3f}")
print("\nO eixo do topo é onde a decisão de projeto pesa mais — e onde vale")
print("gastar discussão. O do fim pode ser escolhido por conveniência.")

## 8. O custo do vazamento

`MEDIA_DIAS_ATRASO_PAG` e `MEDIA_DIAS_ATRASO_COM` compartilham origem aritmética
com o alvo: são médias de atraso, e o alvo é uma taxa de atraso. Elas ficaram no
modelo conscientemente — a decisão está registrada no notebook 04.

Esta seção põe um número nessa decisão. A diferença de MCC entre o conjunto
completo e o mesmo cenário sem essas duas colunas é **a medida do vazamento**: o
quanto do desempenho vem de recalcular a própria regra em vez de prever conduta.

In [ ]:
pivo = modelos_reais.pivot_table(
    index=["populacao", "alvo", "modelo"], columns="features",
    values=METRICA_RANKING,
)

if {"completo", "sem MEDIA_DIAS_ATRASO"}.issubset(pivo.columns):
    pivo = pivo.dropna(subset=["completo", "sem MEDIA_DIAS_ATRASO"])
    pivo["queda"] = pivo["completo"] - pivo["sem MEDIA_DIAS_ATRASO"]
    pivo["queda_%"] = pivo["queda"] / pivo["completo"] * 100

    print(f"Queda média de MCC ao remover as duas colunas: "
          f"{pivo['queda'].mean():.3f} ({pivo['queda_%'].mean():.0f}%)")
    print(f"Maior queda observada: {pivo['queda'].max():.3f}")
    print(f"Menor queda observada: {pivo['queda'].min():.3f}")

    print("\nOnde o vazamento mais pesa:")
    piores = pivo.nlargest(8, "queda").reset_index()
    print(f"  {'população':<20}{'alvo':<20}{'modelo':<22}"
          f"{'completo':>9}{'sem':>8}{'queda':>8}")
    for _, r in piores.iterrows():
        print(f"  {r['populacao']:<20}{r['alvo']:<20}{r['modelo']:<22}"
              f"{r['completo']:>9.3f}{r['sem MEDIA_DIAS_ATRASO']:>8.3f}{r['queda']:>8.3f}")

    melhor_limpo = (modelos_reais[modelos_reais["features"] != "completo"]
                    .nlargest(1, METRICA_RANKING).iloc[0])
    print(f"\nMelhor cenário SEM as colunas de vazamento:")
    print(f"  {melhor_limpo['populacao']} · {melhor_limpo['alvo']} · "
          f"{melhor_limpo['features']} · {melhor_limpo['modelo']}")
    print(f"  MCC {melhor_limpo[METRICA_RANKING]:.3f} · AUC {melhor_limpo['roc_auc']:.3f}")
    print("\nEste é o número honesto para citar como capacidade preditiva:")
    print("o conjunto completo mede a aderência à regra, não a previsão de conduta.")
else:
    print("Cenários de ablação não presentes na grade — acrescente-os em "
          "CENARIOS_FEATURES para medir o vazamento.")

## 9. A grade, vista de uma vez

Dois gráficos. O primeiro mostra a dispersão dentro de cada eixo — barras longas
são decisões que importam. O segundo mostra o efeito conjunto de população e
alvo sobre o melhor modelo de cada célula.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.6))

# ── Esquerda: dispersão do MCC por eixo ──────────────────────────────
posicao, rotulos, dados_caixa = [], [], []
CORES_EIXO = {"populacao": "#2a78d6", "alvo": "#eb6834",
              "features": "#1baf7a", "modelo": "#8b5cf6"}
cores_caixa = []

for eixo in ["populacao", "alvo", "features", "modelo"]:
    for nome, grupo in modelos_reais.groupby(eixo):
        dados_caixa.append(grupo[METRICA_RANKING].values)
        rotulos.append(str(nome)[:22])
        cores_caixa.append(CORES_EIXO[eixo])

caixas = eixos[0].boxplot(dados_caixa, vert=False, patch_artist=True,
                          widths=0.6, medianprops={"color": "#0b0b0b"})
for caixa, cor in zip(caixas["boxes"], cores_caixa):
    caixa.set_facecolor(cor)
    caixa.set_alpha(0.65)
    caixa.set_edgecolor(cor)

eixos[0].set_yticklabels(rotulos, fontsize=8)
eixos[0].set_xlabel("MCC (validação cruzada)")
eixos[0].set_title("Dispersão do MCC dentro de cada eixo", fontsize=11, fontweight="bold")
eixos[0].axvline(0, color="#b8b7b2", linestyle="--", linewidth=1)

marcadores = [plt.Line2D([0], [0], color=c, linewidth=6, alpha=0.65)
              for c in CORES_EIXO.values()]
eixos[0].legend(marcadores, list(CORES_EIXO), fontsize=8, loc="lower right",
                title="eixo", title_fontsize=8)

# ── Direita: melhor MCC por população × alvo ─────────────────────────
mapa = modelos_reais.pivot_table(index="populacao", columns="alvo",
                                 values=METRICA_RANKING, aggfunc="max")
imagem = eixos[1].imshow(mapa.values, cmap="Blues", aspect="auto",
                         vmin=0, vmax=max(0.01, float(np.nanmax(mapa.values))))
eixos[1].set_xticks(range(len(mapa.columns)))
eixos[1].set_xticklabels(mapa.columns, rotation=20, ha="right", fontsize=8)
eixos[1].set_yticks(range(len(mapa.index)))
eixos[1].set_yticklabels(mapa.index, fontsize=8)
eixos[1].set_title("Melhor MCC por população × alvo", fontsize=11, fontweight="bold")
eixos[1].grid(False)

for i in range(mapa.shape[0]):
    for j in range(mapa.shape[1]):
        valor = mapa.values[i, j]
        if np.isnan(valor):
            eixos[1].text(j, i, "—", ha="center", va="center",
                          color="#b8b7b2", fontsize=9)
        else:
            limite = float(np.nanmax(mapa.values)) * 0.6
            eixos[1].text(j, i, f"{valor:.2f}", ha="center", va="center",
                          fontsize=8.5,
                          color="white" if valor > limite else "#0b0b0b")

plt.colorbar(imagem, ax=eixos[1], fraction=0.046, pad=0.04, label="MCC")
plt.tight_layout()
plt.show()

print("Célula vazia (—): cenário inviável, classe rara pequena demais para validar.")

## 10. A resposta

A pergunta que abre este notebook — *quais parâmetros, variáveis e
hiperparâmetros geram o melhor resultado* — tem três respostas, não uma, porque
"melhor" depende do que se pretende fazer com o modelo.

In [ ]:
def descrever(linha, titulo):
    print(f"\n{titulo}")
    print("-" * len(titulo))
    print(f"  população : {linha['populacao']}  ({linha['n_clientes']:,} clientes, "
          f"prevalência {linha['prevalencia']:.1%})")
    print(f"  alvo      : {linha['alvo']}")
    print(f"  features  : {linha['features']} ({int(linha['n_features'])} colunas)")
    print(f"  modelo    : {linha['modelo']}")
    print(f"  MCC {linha[METRICA_RANKING]:.3f} (±{linha[METRICA_RANKING + '_dp']:.3f}) · "
          f"bal.acc {linha['balanced_accuracy']:.3f} · AUC {linha['roc_auc']:.3f} · "
          f"recall {linha['recall']:.3f}")

print("=" * 78)
print(f"{'RESPOSTA DA GRADE':^78}")
print("=" * 78)

campeao_geral = modelos_reais.nlargest(1, METRICA_RANKING).iloc[0]
descrever(campeao_geral, "1. Melhor resultado absoluto")
print("     Use para: demonstrar o teto do que a base permite.")

sem_vazamento = modelos_reais[modelos_reais["features"] != "completo"]
if not sem_vazamento.empty:
    campeao_limpo = sem_vazamento.nlargest(1, METRICA_RANKING).iloc[0]
    descrever(campeao_limpo, "2. Melhor resultado sem as colunas de vazamento")
    print("     Use para: estimar a capacidade preditiva real. É o número honesto.")

# Cobertura máxima: a população que retém MAIS clientes, sem as colunas de
# vazamento. É o cenário de quem precisa escorar a carteira toda, não só quem
# tem histórico longo.
sem_corte = modelos_reais[modelos_reais["features"] != "completo"]
if not sem_corte.empty:
    maior_cobertura = sem_corte["n_clientes"].max()
    abrangente = sem_corte[sem_corte["n_clientes"] == maior_cobertura]
    campeao_op = abrangente.nlargest(1, METRICA_RANKING).iloc[0]
    descrever(campeao_op, "3. Melhor resultado com cobertura máxima, sem vazamento")
    print("     Use para: escorar todo mundo, inclusive cliente de histórico curto.")

print("\n" + "=" * 78)
print("Os três são resultados legítimos de perguntas diferentes. Citar o primeiro")
print("como capacidade preditiva seria erro; ignorá-lo, também — ele mede o quanto")
print("o modelo reproduz a regra de negócio, que é uma pergunta válida.")

## 11. Estabilidade do campeão

Um MCC alto obtido num cenário de classe rara pequena é frágil: o desvio entre os
folds diz quanto do resultado é sorteio.

A regra prática: quando o desvio se aproxima da diferença entre dois cenários,
eles são empate técnico — e a escolha entre eles passa a ser de negócio, não de
métrica.

In [ ]:
top = modelos_reais.nlargest(10, METRICA_RANKING).copy()
top["intervalo"] = top[METRICA_RANKING + "_dp"] * 1.96 / np.sqrt(CV["n_splits"])

print(f"{'cenário':<58}{'MCC':>7}{'± IC95':>9}{'raros':>7}")
print("-" * 81)
for _, r in top.iterrows():
    etiqueta = f"{r['populacao']} · {r['modelo']} · {r['features']}"
    print(f"{etiqueta[:57]:<58}{r[METRICA_RANKING]:>7.3f}"
          f"{r['intervalo']:>9.3f}{int(r['classe_rara']):>7}")

primeiro, segundo = top.iloc[0], top.iloc[1]
diferenca = primeiro[METRICA_RANKING] - segundo[METRICA_RANKING]
margem = primeiro["intervalo"] + segundo["intervalo"]

print()
if diferenca < margem:
    print(f"1º e 2º diferem {diferenca:.3f}, dentro da margem conjunta ({margem:.3f}):")
    print("EMPATE TÉCNICO. A escolha entre eles é de negócio — simplicidade,")
    print("interpretabilidade, cobertura da carteira — não de métrica.")
else:
    print(f"1º supera o 2º em {diferenca:.3f}, acima da margem conjunta ({margem:.3f}):")
    print("a diferença sobrevive à incerteza da validação cruzada.")

## 9. O pacote de produção

Este é o **único** ponto do pipeline que produz um modelo para uso real.

Os notebooks 05-07 respondem *"qual hiperparâmetro é melhor neste cenário"* — e o
campeão de um estudo de hiperparâmetro é o melhor ajuste dentro de um cenário que
pode não ser o melhor cenário. Como a seção 6 mostra, o eixo de features costuma
mover o MCC mais do que o algoritmo. Promover o vencedor de um estudo ignoraria
essa evidência.

Aqui, com os quatro eixos varridos, existe base para escolher. O pacote sai em
dois formatos, por motivos diferentes:

| arquivo | para quê |
|:---|:---|
| `.pkl` | o pipeline treinado — pré-processamento e modelo juntos, pronto para `predict_proba` |
| `.parquet` | a carteira inteira já escorada, para consumo em BI/SQL sem carregar Python |

O modelo é ajustado no **treino** do cenário campeão e medido no **teste** que
nunca viu — mesma partição 70/30 do notebook 04, para que a métrica seja
comparável à dos estudos. Só depois de medido ele é re-treinado em 100% dos dados
para o artefato final: a métrica vem do holdout, o modelo entregue aproveita
todos os dados disponíveis.


In [ ]:
if not EMPACOTAR_CAMPEAO:
    print("EMPACOTAR_CAMPEAO = False — nenhum pacote de produção gerado.")
else:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (matthews_corrcoef, balanced_accuracy_score,
                                 roc_auc_score, recall_score, precision_score, f1_score)

    # ── 1. Qual linha da grade vira produção ─────────────────────────────
    sem_vaz = modelos_reais[modelos_reais["features"] != "completo"]
    if CAMPEAO_PRODUCAO == "absoluto":
        escolhido = modelos_reais.nlargest(1, METRICA_RANKING).iloc[0]
    elif CAMPEAO_PRODUCAO == "cobertura_maxima":
        if sem_vaz.empty:
            raise ValueError("Sem cenários sem vazamento na grade.")
        abrangente = sem_vaz[sem_vaz["n_clientes"] == sem_vaz["n_clientes"].max()]
        escolhido = abrangente.nlargest(1, METRICA_RANKING).iloc[0]
    elif CAMPEAO_PRODUCAO == "sem_vazamento":
        if sem_vaz.empty:
            raise ValueError(
                "CAMPEAO_PRODUCAO='sem_vazamento' exige ao menos um conjunto de "
                "features que remova MEDIA_DIAS_ATRASO — acrescente em CENARIOS_FEATURES."
            )
        escolhido = sem_vaz.nlargest(1, METRICA_RANKING).iloc[0]
    else:
        raise ValueError(f"CAMPEAO_PRODUCAO='{CAMPEAO_PRODUCAO}' inválido.")

    # ── 2. Reconstrói exatamente aquele cenário ──────────────────────────
    cen_pop = next(p for p in CENARIOS_POPULACAO if p["nome"] == escolhido["populacao"])
    cen_alvo = next(a for a in CENARIOS_ALVO if a["nome"] == escolhido["alvo"])
    cen_feat = next(t for t in CENARIOS_FEATURES if t["nome"] == escolhido["features"])
    cen_modelo = next(m for m in CENARIOS_MODELO if m["nome"] == escolhido["modelo"])

    universo_prod = aplicar_populacao(dados, cen_pop)
    y_prod = construir_alvo(universo_prod, cen_alvo)
    num_prod, cat_prod = montar_features(cen_feat)
    COLS_PROD = num_prod + cat_prod
    X_prod = universo_prod[COLS_PROD]

    print("=" * 74)
    print(f"{'CENÁRIO PROMOVIDO A PRODUÇÃO':^74}")
    print("=" * 74)
    print(f"  critério  : {CAMPEAO_PRODUCAO}")
    print(f"  população : {cen_pop['nome']}  ({len(universo_prod):,} clientes)")
    print(f"  alvo      : {cen_alvo['nome']}  (prevalência {y_prod.mean():.1%})")
    print(f"  features  : {cen_feat['nome']}  ({len(COLS_PROD)} colunas)")
    print(f"  modelo    : {cen_modelo['nome']}")
    print(f"  MCC na CV : {escolhido[METRICA_RANKING]:.3f} "
          f"(±{escolhido[METRICA_RANKING + '_dp']:.3f})")

    # ── 3. Holdout: mede no que o modelo não viu ─────────────────────────
    Xtr, Xte, ytr, yte = train_test_split(
        X_prod, y_prod, test_size=TEST_SIZE_PRODUCAO,
        random_state=RANDOM_STATE, stratify=y_prod,
    )
    pipe_prod = montar_pipeline(instanciar(cen_modelo), num_prod, cat_prod)
    pipe_prod.fit(Xtr, ytr)
    prob_te = pipe_prod.predict_proba(Xte)[:, 1]
    pred_te = (prob_te >= 0.50).astype(int)

    METRICAS_TESTE = {
        "mcc": float(matthews_corrcoef(yte, pred_te)),
        "balanced_accuracy": float(balanced_accuracy_score(yte, pred_te)),
        "roc_auc": float(roc_auc_score(yte, prob_te)),
        "recall": float(recall_score(yte, pred_te, zero_division=0)),
        "precision": float(precision_score(yte, pred_te, zero_division=0)),
        "f1": float(f1_score(yte, pred_te, zero_division=0)),
    }
    print(f"\n  holdout ({len(yte):,} clientes, {int(yte.sum())} da classe positiva):")
    for k, v in METRICAS_TESTE.items():
        print(f"    {k:<20}{v:.3f}")
    if min(int(yte.sum()), int((yte == 0).sum())) < 20:
        print("\n  ATENÇÃO: classe rara pequena no holdout — a métrica de CV acima")
        print("  é mais confiável que a de teste para este cenário.")

    # ── 4. Re-treina em 100% para o artefato ─────────────────────────────
    # A métrica já foi medida; o modelo entregue aproveita todos os dados.
    pipe_prod = montar_pipeline(instanciar(cen_modelo), num_prod, cat_prod)
    pipe_prod.fit(X_prod, y_prod)
    print(f"\n  modelo final re-treinado em {len(X_prod):,} clientes.")


In [ ]:
if EMPACOTAR_CAMPEAO:
    import pickle
    from datetime import datetime, timezone

    MODEL_CARD = {
        "origem": "notebook 08 — varredura de cenários",
        "criterio_selecao": CAMPEAO_PRODUCAO,
        "cenario": {
            "populacao": cen_pop["nome"], "populacao_regra": cen_pop,
            "alvo": cen_alvo["nome"], "alvo_regra": cen_alvo,
            "features_conjunto": cen_feat["nome"],
            "features_removidas": cen_feat["remover"],
        },
        "algoritmo": cen_modelo["classe"].__name__,
        "hiperparametros": cen_modelo["hp"],
        "features": COLS_PROD,
        "features_numericas": num_prod, "features_categoricas": cat_prod,
        "target": "ALTO_RISCO",
        "n_treino_final": int(len(X_prod)),
        "prevalencia": float(y_prod.mean()),
        "threshold_classificacao": 0.50,
        "metricas_cv": {m: float(escolhido[m]) for m in SCORING if m in escolhido},
        "metricas_holdout": METRICAS_TESTE,
        "data_version": DATA_VERSION, "random_state": RANDOM_STATE,
        "gerado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

    # Carteira escorada: o universo de negócio INTEIRO, não só a população do
    # cenário. Quem ficou fora do recorte não treinou o modelo, mas ainda pode
    # ser escorado por ele — é justamente para isso que o pacote existe.
    escorar = dados[dados["CORE_BUSINESS"] == 1] if cen_pop["core"] else dados
    escorados = escorar[["ID_PESSOA"]].copy()
    escorados["PROB_RISCO"] = pipe_prod.predict_proba(escorar[COLS_PROD])[:, 1]
    escorados["FAIXA_RISCO"] = pd.cut(
        escorados["PROB_RISCO"], bins=[-0.001, 0.35, 0.65, 1.001],
        labels=["BAIXO", "MÉDIO", "ALTO"],
    ).astype(str)
    escorados["_DATA_VERSION"] = DATA_VERSION
    escorados["_NO_TREINO"] = escorados["ID_PESSOA"].isin(universo_prod["ID_PESSOA"])

    SUFIXO = DATA_VERSION.replace(".", "_")
    if EM_DATABRICKS:
        if MLFLOW_ATIVO:
            from mlflow.models import infer_signature
            if mlflow.active_run() is not None:
                mlflow.end_run()
            mlflow.set_experiment(f"{MLFLOW_RAIZ}__PRODUCAO")
            with mlflow.start_run(run_name=f"campeao__{CAMPEAO_PRODUCAO}__v{DATA_VERSION}") as run:
                mlflow.set_tags({
                    "eixo.populacao": cen_pop["nome"], "eixo.alvo": cen_alvo["nome"],
                    "eixo.features": cen_feat["nome"], "eixo.modelo": cen_modelo["nome"],
                    "criterio": CAMPEAO_PRODUCAO, "pacote": "producao",
                })
                mlflow.log_params({f"hp.{k}": v for k, v in cen_modelo["hp"].items()})
                mlflow.log_metrics({f"teste.{k}": v for k, v in METRICAS_TESTE.items()})
                mlflow.log_metrics({f"cv.{m}": float(escolhido[m])
                                    for m in SCORING if m in escolhido})
                mlflow.log_dict(MODEL_CARD, "model_card.json")
                assinatura = infer_signature(X_prod, pipe_prod.predict(X_prod))
                mlflow.sklearn.log_model(sk_model=pipe_prod, name="modelo",
                                         signature=assinatura, input_example=X_prod.head(3))
                try:
                    mlflow.register_model(f"runs:/{run.info.run_id}/modelo", NOME_REGISTRADO)
                    print(f"Registrado em {NOME_REGISTRADO}")
                except Exception as e:
                    print(f"Registro falhou ({type(e).__name__}); modelo em "
                          f"runs:/{run.info.run_id}/modelo")
                print(f"Run de produção: {run.info.run_id}")
        destino = f"{TABELA.rsplit('.', 1)[0]}.carteira_escorada_v{SUFIXO}"
        (spark.createDataFrame(escorados)                          # noqa: F821
            .write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true").saveAsTable(destino))
        print(f"Carteira escorada: {destino}")
    else:
        caminho_pkl = OUTPUT_DIR / f"modelo_campeao_v{SUFIXO}.pkl"
        caminho_pq = OUTPUT_DIR / f"carteira_escorada_v{SUFIXO}.parquet"
        if (caminho_pkl.exists() or caminho_pq.exists()) and not SOBRESCREVER:
            raise FileExistsError(
                f"{caminho_pkl.name} ou {caminho_pq.name} já existe. "
                "Incremente DATA_VERSION ou use SOBRESCREVER = True."
            )
        with open(caminho_pkl, "wb") as arq:
            pickle.dump({"pipeline": pipe_prod, "model_card": MODEL_CARD}, arq)
        print(f"{caminho_pkl.name}   pipeline + model_card")

        # Parquet exige pyarrow (ou fastparquet). Se faltar, o CSV equivalente
        # sai no lugar — o pacote não fica pela metade por causa de dependência.
        try:
            escorados.to_parquet(caminho_pq, index=False)
            print(f"{caminho_pq.name}   {len(escorados):,} clientes escorados")
        except ImportError:
            caminho_csv_esc = caminho_pq.with_suffix(".csv")
            escorados.to_csv(caminho_csv_esc, **CSV_SAIDA)
            print(f"{caminho_csv_esc.name}   {len(escorados):,} clientes escorados")
            print("  (parquet indisponível: pip install pyarrow para gerar .parquet)")

    print()
    print(escorados["FAIXA_RISCO"].value_counts()
          .reindex(["BAIXO", "MÉDIO", "ALTO"]).to_string())
    print("\nPara escorar novos clientes:")
    print("  prob = pipeline.predict_proba(df[model_card['features']])[:, 1]")


## 12. Publicação

Grava a grade inteira — viáveis e inviáveis, com as métricas e o motivo do
descarte. É este arquivo que sustenta a comparação de cenários no relatório, e
é ele que permite refazer a leitura sem reprocessar nada.

In [ ]:
COLUNAS_SAIDA = [
    "populacao", "alvo", "features", "modelo",
    "n_clientes", "n_features", "prevalencia", "classe_rara", "raros_por_fold",
    "viavel", "motivo",
] + [c for m in SCORING for c in (m, f"{m}_dp")]

saida = resultados.reindex(columns=COLUNAS_SAIDA)
saida.insert(0, "_DATA_VERSION", DATA_VERSION)
saida = saida.sort_values(METRICA_RANKING, ascending=False, na_position="last")

if EM_DATABRICKS:
    destino = f"{TABELA.rsplit('.', 1)[0]}.cenarios_v{DATA_VERSION.replace('.', '_')}"
    (spark.createDataFrame(saida)                              # noqa: F821
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(destino))
    print(f"Tabela publicada: {destino}")
else:
    if CSV_DESTINO.exists() and not SOBRESCREVER:
        raise FileExistsError(
            f"{CSV_DESTINO.name} já existe. Incremente DATA_VERSION ou use "
            "SOBRESCREVER = True."
        )
    saida.to_csv(CSV_DESTINO, **CSV_SAIDA)
    print(f"{CSV_DESTINO}")

print(f"{len(saida):,} cenários × {saida.shape[1]} colunas")
print(f"  {int(saida['viavel'].sum()):,} avaliados · "
      f"{int((~saida['viavel']).sum())} inviáveis registrados")